# Data Understanding

## Mục tiêu

Hiểu cấu trúc bộ dữ liệu H&M Personalized Fashion Recommendations trước khi thực hiện Data Cleaning và Data Modeling.

Các nội dung sẽ thực hiện:

- Khám phá số lượng bản ghi
- Khám phá schema
- Kiểm tra kiểu dữ liệu
- Kiểm tra dữ liệu thiếu
- Xác nhận grain của từng bảng
- Ghi nhận các vấn đề dữ liệu

In [5]:
from pathlib import Path
import duckdb

# ==============================
# Project Paths
# ==============================

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

DOCS_DIR = PROJECT_ROOT / "docs"
REPORTS_DIR = PROJECT_ROOT / "reports"

# ==============================
# DuckDB Connection
# ==============================

con = duckdb.connect()

print("Project Root :", PROJECT_ROOT)
print("Raw Data     :", RAW_DIR)

Project Root : d:\MindX_3\Final_Project\hm-fashion-analytics
Raw Data     : d:\MindX_3\Final_Project\hm-fashion-analytics\data\raw


## 1. Khởi tạo môi trường

## 2. Data Profiling

## 2.1 Số lượng bản ghi của từng bảng

### Mục tiêu

Xác định quy mô của từng bảng trong bộ dữ liệu trước khi tiến hành các bước phân tích tiếp theo.

In [7]:
import duckdb

con = duckdb.connect()

tables = [
    ("customers", "customers.csv"),
    ("articles", "articles.csv"),
    ("transactions", "transactions_train.csv"),
]

for table_name, file_name in tables:
    file_path = RAW_DIR / file_name

    row_count = con.execute(
        f"SELECT COUNT(*) FROM read_csv_auto('{file_path.as_posix()}')"
    ).fetchone()[0]

    print(f"{table_name}: {row_count:,} rows")

customers: 1,371,980 rows
articles: 105,542 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

transactions: 31,788,324 rows


### Nhận xét

- Bộ dữ liệu gồm 3 bảng: customers, articles và transactions.
- Bảng transactions có quy mô lớn nhất với hơn 31 triệu bản ghi.
- Bảng customers có khoảng 1.37 triệu khách hàng.
- Bảng articles có hơn 105 nghìn sản phẩm.
- Kết quả này cho thấy việc sử dụng DuckDB là phù hợp để xử lý dữ liệu thay vì đọc toàn bộ bằng pandas.

## 2.2 Khám phá cấu trúc bảng (Schema)

### Mục tiêu

Xác định các cột trong từng bảng và kiểu dữ liệu mà DuckDB tự động suy luận khi đọc file CSV.

In [8]:
for table_name, file_name in tables:

    print("=" * 70)
    print(f"Schema của bảng: {table_name}")
    print("=" * 70)

    file_path = RAW_DIR / file_name

    schema = con.execute(
        f"""
        DESCRIBE
        SELECT *
        FROM read_csv_auto('{file_path.as_posix()}')
        """
    ).fetchdf()

    display(schema)

Schema của bảng: customers


,column_name,column_type,null,key,default,extra
0,customer_id,VARCHAR,YES,None,None,None
1,FN,DOUBLE,YES,None,None,None
2,Active,DOUBLE,YES,None,None,None
3,club_member_status,VARCHAR,YES,None,None,None
4,fashion_news_frequency,VARCHAR,YES,None,None,None
5,age,BIGINT,YES,None,None,None
6,postal_code,VARCHAR,YES,None,None,None


Schema của bảng: articles


,column_name,column_type,null,key,default,extra
0,article_id,VARCHAR,YES,None,None,None
1,product_code,VARCHAR,YES,None,None,None
2,prod_name,VARCHAR,YES,None,None,None
3,product_type_no,BIGINT,YES,None,None,None
4,product_type_name,VARCHAR,YES,None,None,None
5,product_group_name,VARCHAR,YES,None,None,None
6,graphical_appearance_no,BIGINT,YES,None,None,None
7,graphical_appearance_name,VARCHAR,YES,None,None,None
8,colour_group_code,VARCHAR,YES,None,None,None
9,colour_group_name,VARCHAR,YES,None,None,None


Schema của bảng: transactions


,column_name,column_type,null,key,default,extra
0,t_dat,DATE,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,article_id,VARCHAR,YES,None,None,None
3,price,DOUBLE,YES,None,None,None
4,sales_channel_id,BIGINT,YES,None,None,None


## 2.3 Kiểm tra giá trị thiếu (Null Values)

### Mục tiêu

Xác định số lượng và tỷ lệ giá trị thiếu của từng cột trong mỗi bảng để phục vụ cho giai đoạn Data Cleaning.

In [10]:
def null_report(file_name, table_name):
    file_path = RAW_DIR / file_name

    # Lấy danh sách cột
    columns = con.execute(
        f"""
        DESCRIBE
        SELECT *
        FROM read_csv_auto('{file_path.as_posix()}')
        """
    ).fetchdf()["column_name"].tolist()

    # Tổng số dòng
    total_rows = con.execute(
        f"""
        SELECT COUNT(*)
        FROM read_csv_auto('{file_path.as_posix()}')
        """
    ).fetchone()[0]

    print(f"\n{'='*70}")
    print(f"Null Report - {table_name}")
    print(f"{'='*70}")

    for column in columns:
        null_count = con.execute(
            f"""
            SELECT COUNT(*)
            FROM read_csv_auto('{file_path.as_posix()}')
            WHERE "{column}" IS NULL
            """
        ).fetchone()[0]

        if null_count > 0:
            percent = null_count / total_rows * 100

            print(
                f"{column:<35} "
                f"{null_count:>10,} "
                f"({percent:.2f}%)"
            )

In [11]:
null_report("customers.csv", "customers")
null_report("articles.csv", "articles")
null_report("transactions_train.csv", "transactions")


Null Report - customers
FN                                     895,050 (65.24%)
Active                                 907,576 (66.15%)
club_member_status                       6,062 (0.44%)
fashion_news_frequency                  16,009 (1.17%)
age                                     15,861 (1.16%)

Null Report - articles
detail_desc                                416 (0.39%)

Null Report - transactions


## 2.4 Cardinality

### Mục tiêu

Kiểm tra số lượng giá trị duy nhất (distinct values) của các khóa chính và khóa ngoại quan trọng trong bộ dữ liệu.

In [12]:
checks = [
    ("customers.csv", "customer_id"),
    ("articles.csv", "article_id"),
    ("transactions_train.csv", "customer_id"),
    ("transactions_train.csv", "article_id"),
]

print("=" * 70)
print("CARDINALITY")
print("=" * 70)

for file_name, column in checks:
    file_path = RAW_DIR / file_name

    distinct_count = con.execute(
        f"""
        SELECT COUNT(DISTINCT "{column}")
        FROM read_csv_auto('{file_path.as_posix()}')
        """
    ).fetchone()[0]

    print(f"{file_name:<25} | {column:<15} : {distinct_count:,}")

CARDINALITY
customers.csv             | customer_id     : 1,371,980
articles.csv              | article_id      : 105,542
transactions_train.csv    | customer_id     : 1,362,281
transactions_train.csv    | article_id      : 104,547


## 2.5 Kiểm tra Grain của bảng transactions

### Mục tiêu

Xác nhận mức độ chi tiết (grain) của bảng `transactions_train` bằng cách kiểm tra tính duy nhất của khóa tổng hợp.

Giả thuyết ban đầu:

(customer_id, article_id, t_dat, sales_channel_id)

In [13]:
file_path = RAW_DIR / "transactions_train.csv"

# Tổng số dòng
total_rows = con.execute(
    f"""
    SELECT COUNT(*)
    FROM read_csv_auto('{file_path.as_posix()}')
    """
).fetchone()[0]

# Số dòng duy nhất theo khóa tổng hợp
distinct_rows = con.execute(
    f"""
    SELECT COUNT(*)
    FROM (
        SELECT DISTINCT
            customer_id,
            article_id,
            t_dat,
            sales_channel_id
        FROM read_csv_auto('{file_path.as_posix()}')
    )
    """
).fetchone()[0]

print(f"Tổng số dòng                : {total_rows:,}")
print(f"Số dòng distinct composite  : {distinct_rows:,}")
print(f"Số dòng trùng              : {total_rows - distinct_rows:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Tổng số dòng                : 31,788,324
Số dòng distinct composite  : 28,583,889
Số dòng trùng              : 3,204,435


## 2.6 Date Range

In [14]:
file_path = RAW_DIR / "transactions_train.csv"

min_date, max_date = con.execute(
    f"""
    SELECT
        MIN(t_dat),
        MAX(t_dat)
    FROM read_csv_auto('{file_path.as_posix()}')
    """
).fetchone()

print(f"Từ ngày : {min_date}")
print(f"Đến ngày: {max_date}")

Từ ngày : 2018-09-20
Đến ngày: 2020-09-22


## 2.7 Sample Data

In [15]:
display(
    con.execute(
        f"""
        SELECT *
        FROM read_csv_auto('{(RAW_DIR / 'customers.csv').as_posix()}')
        LIMIT 5
        """
    ).fetchdf()
)

display(
    con.execute(
        f"""
        SELECT *
        FROM read_csv_auto('{(RAW_DIR / 'articles.csv').as_posix()}')
        LIMIT 5
        """
    ).fetchdf()
)

display(
    con.execute(
        f"""
        SELECT *
        FROM read_csv_auto('{(RAW_DIR / 'transactions_train.csv').as_posix()}')
        LIMIT 5
        """
    ).fetchdf()
)

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,NaN,NaN,ACTIVE,NONE,54,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1.0,1.0,ACTIVE,Regularly,52,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,0108775015,0108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,09,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,0108775044,0108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,0108775051,0108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,0110065001,0110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,09,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,0110065002,0110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,0685687004,0.016932,2
